In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import sys
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.getcwd())

# 1. Tải và chia dữ liệu
df = pd.read_csv("data/processed/featured_reviews.csv")
df = df.dropna(subset=['reviews.rating', 'Cleaned_Review']).copy()
df['Sentiment'] = df['reviews.rating'].apply(lambda x: 1 if float(x) >= 4 else 0)

X_train, X_test, y_train, y_test = train_test_split(df['Cleaned_Review'], df['Sentiment'], test_size=0.2, random_state=42, stratify=df['Sentiment'])

# 2. Huấn luyện lại mô hình mạnh nhất (Logistic Regression) để lấy dự đoán
vec = TfidfVectorizer(max_features=2000, stop_words='english')
X_train_vec = vec.fit_transform(X_train.fillna(''))
X_test_vec = vec.transform(X_test.fillna(''))

clf = LogisticRegression(max_iter=1000, random_state=42, n_jobs=4)
clf.fit(X_train_vec, y_train)
y_pred = clf.predict(X_test_vec)

# 3. Trích xuất các ca dự đoán sai (Error Analysis)
test_df = pd.DataFrame({'Review_Text': X_test, 'Actual': y_test, 'Predicted': y_pred})
errors = test_df[test_df['Actual'] != test_df['Predicted']]

print("=== PHÂN TÍCH LỖI DỰ ĐOÁN (ERROR ANALYSIS) ===")
false_positives = errors[(errors['Actual'] == 0) & (errors['Predicted'] == 1)]
print(f"\n1. Nhận diện sai thành KHEN (False Positive): {len(false_positives)} ca")
print("=> Lý do phổ biến: Khách hàng dùng từ ngữ tích cực (tốt, đẹp) nhưng để mỉa mai (sarcasm) hoặc khen một điểm nhưng chê thậm tệ điểm khác.")
for rev in false_positives['Review_Text'].head(3).values:
    print(f"   - {rev[:120]}...")

false_negatives = errors[(errors['Actual'] == 1) & (errors['Predicted'] == 0)]
print(f"\n2. Nhận diện sai thành CHÊ (False Negative): {len(false_negatives)} ca")
print("=> Lý do phổ biến: Khách hàng chấm 4-5 sao nhưng văn phong lại dùng nhiều từ mang tính phủ định (không tệ, chưa từng thấy...).")
for rev in false_negatives['Review_Text'].head(3).values:
    print(f"   - {rev[:120]}...")

=== PHÂN TÍCH LỖI DỰ ĐOÁN (ERROR ANALYSIS) ===

1. Nhận diện sai thành KHEN (False Positive): 172 ca
=> Lý do phổ biến: Khách hàng dùng từ ngữ tích cực (tốt, đẹp) nhưng để mỉa mai (sarcasm) hoặc khen một điểm nhưng chê thậm tệ điểm khác.
   - hotel easy find everything would expect budget chain like motel free parking wifi clean rooms although bathrooms could u...
   - platinum marriott rewards member stayed july fourth july weekend ran peachtree roadrace k race location stay great howev...
   - stayed business nice enough staff stayed storm blew awakened wind whistling terrace door tried stuffing towels gap makin...

2. Nhận diện sai thành CHÊ (False Negative): 38 ca
=> Lý do phổ biến: Khách hàng chấm 4-5 sao nhưng văn phong lại dùng nhiều từ mang tính phủ định (không tệ, chưa từng thấy...).
   - stayed summer thought put review hotel room rated average price frills hoteli admit first arrived del sol inn impressed ...
   - rooms property need updating rooms small quiet airconditioning

c:\Users\Pangorin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=4', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
